In [1]:
import torch
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [8]:
df = pd.read_csv('housing_prices.csv')
df.head()

,price,area,mainroad,neighborhood,airconditioning,stories
0,13300000,7420,yes,A,yes,3
1,12250000,8960,yes,A,yes,4
2,12250000,9960,yes,B,no,2
3,12215000,7500,yes,A,yes,2
4,11410000,7420,yes,A,yes,2


In [9]:
X = df.drop(columns=['price'])
y = df['price']

In [11]:
X['mainroad'] = X['mainroad'].map({'no':0, 'yes':1})
X['airconditioning'] = X['airconditioning'].map({'no':0, 'yes':1})

In [12]:
X = pd.get_dummies(X,columns=['neighborhood'],dtype=int)

In [13]:
print(X.head())

   area  mainroad  ...  neighborhood_B  neighborhood_C
0  7420         1  ...               0               0
1  8960         1  ...               0               0
2  9960         1  ...               1               0
3  7500         1  ...               0               0
4  7420         1  ...               0               0

[5 rows x 7 columns]


In [14]:
print(X.columns.tolist())

['area', 'mainroad', 'airconditioning', 'stories', 'neighborhood_A', 'neighborhood_B', 'neighborhood_C']


In [15]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [16]:
scaler = StandardScaler()

In [17]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [19]:
y_train = scaler.fit_transform(y_train.to_numpy().reshape(-1,1))
y_test = scaler.transform(y_test.to_numpy().reshape(-1,1))

In [20]:
X_train = torch.tensor(X_train,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)

y_train = torch.tensor(y_train,dtype=torch.float32)
y_test = torch.tensor(y_test,dtype=torch.float32)

In [21]:
print("X_train shape: ",X_train.shape)
print("X_test shape: ",X_test.shape)
print("y_train shape: ",y_train.shape)
print("y_test shape: ",y_test.shape)

X_train shape:  torch.Size([381, 7])
X_test shape:  torch.Size([164, 7])
y_train shape:  torch.Size([381, 1])
y_test shape:  torch.Size([164, 1])


In [22]:
import torch.nn as nn

In [23]:
torch.manual_seed(42)

In [24]:
class HousePriceModel(nn.Module):
    def __init__(self,input_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features,64),
            nn.ReLU(),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,1)
        )

    def forward(self,x):
        return self.network(x)

In [26]:
model = HousePriceModel(input_features=X_train.shape[1])
print(model)

HousePriceModel(
  (network): Sequential(
    (0): Linear(in_features=7, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [27]:
sample_input = X_train[:5]

with torch.no_grad():
    predictions = model(sample_input)

print('Predictions: \n',predictions)

print("Actual price: \n",y_train[:5])

Predictions: 
 tensor([[-0.0177],
        [-0.1308],
        [ 0.0317],
        [ 0.0446],
        [ 0.0200]])
Actual price: 
 tensor([[ 0.6305],
        [-0.5938],
        [-0.6333],
        [ 2.0523],
        [ 0.7451]])


In [29]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
epochs=500

for epoch in range(epochs):
    model.train()

    predictions = model(X_train)

    loss = criterion(predictions,y_train)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch+1)%50 == 0:
        print(
            f'Epoch: [{epoch+1}/{epochs}],'
            f'Loss: {loss.item():.4f}'
        )

Epoch: [50/500],Loss: 0.4611
Epoch: [100/500],Loss: 0.4265
Epoch: [150/500],Loss: 0.4114
Epoch: [200/500],Loss: 0.4014
Epoch: [250/500],Loss: 0.3913
Epoch: [300/500],Loss: 0.3810
Epoch: [350/500],Loss: 0.3705
Epoch: [400/500],Loss: 0.3580
Epoch: [450/500],Loss: 0.3440
Epoch: [500/500],Loss: 0.3299


In [30]:
from sklearn.metrics import mean_absolute_error,mean_squared_error

In [32]:
model.eval()

with torch.no_grad():
    test_prediction_scaled = model(X_test).cpu().numpy()

test_prediction = scaler.inverse_transform(test_prediction_scaled).ravel()

y_test = scaler.inverse_transform(y_test.cpu().numpy()).ravel()

mae = mean_absolute_error(y_test,test_prediction)

rmse = np.sqrt(mean_squared_error(y_test,test_prediction))

print("Test set evaluation")
print(f'MAE: {mae:,.2f}')
print(f'RMSE: {rmse:,.2f}')

results = pd.DataFrame({
    "Actual_price": y_test[:10],
    "Predicted_price": test_prediction[:10]
})

results['Absolute_error'] = abs(results['Actual_price']-results['Predicted_price'])

print('\n Sample predictions: ')
print(results.round(2).to_string(index=False))

Test set evaluation
MAE: 1,108,557.88
RMSE: 1,531,306.24

 Sample predictions: 
 Actual_price  Predicted_price  Absolute_error
   4060000.25       4081091.50    2.109125e+04
   6650000.00       6430215.00    2.197850e+05
   3710000.25       4169593.50    4.595935e+05
   6440000.00       5294493.50    1.145506e+06
   2800000.00       3738131.75    9.381320e+05
   4900000.00       4666879.50    2.331205e+05
   5250000.00       6662610.00    1.412610e+06
   4543000.00       4925139.00    3.821390e+05
   2450000.25       2611156.50    1.611562e+05
   3353000.00       2883662.50    4.693378e+05
